In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
import time
import copy
import os

# ==========================================
# 1. SETUP HARDWARE (CUDA / CPU)
# ==========================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 Menjalankan training di perangkat: {DEVICE}")

# ==========================================
# 2. SETUP DATASET & DATALOADER (BERSIH)
# ==========================================
DATA_DIR = 'RespiScan_Train_Ready' # Folder yang baru saja kamu bersihkan

# Transformasi bersih: TANPA rotasi, hanya potong rapi dan flip waktu
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), # Membalik waktu batuk secara acak
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(DATA_DIR, x.upper()), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=0)
    for x in ['train', 'val']
}

# Cek indeks kelas (Biasanya Healthy=0, TB=1 karena urut abjad)
class_names = image_datasets['train'].classes
print(f"📁 Dataset siap! Ditemukan kelas: {class_names} dengan indeks {image_datasets['train'].class_to_idx}")

# ==========================================
# 3. SETUP MOBILENETV2 & OPTIMASI
# ==========================================
model = models.mobilenet_v2(weights='DEFAULT')

# Unfreeze semua layer agar beradaptasi dengan Scalogram
for param in model.parameters():
    param.requires_grad = True

# Modifikasi Kepala Model (Classifier)
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model = model.to(DEVICE)

# Differential Learning Rates
optimizer = optim.Adam([
    {'params': model.features[14:].parameters(), 'lr': 1e-5}, # Badan atas belajar pelan
    {'params': model.classifier.parameters(), 'lr': 1e-4}      # Kepala belajar normal
], lr=1e-5) # Sisa layer bawah menggunakan LR sangat kecil

# Weighted Loss (Asumsi Healthy=0, TB=1. Beri bobot 2.0 untuk Healthy)
weights = torch.tensor([2.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
print("⚖️ Model MobileNetV2 + Weighted Loss (2:1) siap dilatih!")

# ==========================================
# 4. FUNGSI TRAINING
# ==========================================
def train_model_v4(model, criterion, optimizer, num_epochs=30, patience=7):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    early_stop_counter = 0
    
    # Scheduler untuk mengerem Learning Rate jika akurasi nge-stuck
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0
            all_preds, all_labels = [], []

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])
            epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)

            print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f}')

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                history['val_f1'].append(epoch_f1)
                
                scheduler.step(epoch_f1) # Update scheduler

                if epoch_f1 > best_f1:
                    best_f1 = epoch_f1
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), 'best_RespiScan_MobileNetV2_Scalogram.pth')
                    early_stop_counter = 0
                    print("⭐️ Model Terbaik Disimpan!")
                else:
                    early_stop_counter += 1

        if early_stop_counter >= patience:
            print(f"🛑 Early Stopping aktif di epoch {epoch+1}. Model berhenti belajar agar tidak Overfitting.")
            break

    time_elapsed = time.time() - since
    print(f'\n🎯 Training selesai dalam {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'🏆 Best Validation F1-Score: {best_f1:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

# ==========================================
# 5. EKSEKUSI TRAINING
# ==========================================
model_trained, history = train_model_v4(model, criterion, optimizer, num_epochs=30)

KeyboardInterrupt: 

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# ====================================================
# TAMBAHAN WAJIB (Mengembalikan Ingatan Jupyter)
# ====================================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DATA_DIR = 'RespiScan_Train_Ready' # Pastikan nama foldernya benar

data_transforms = {
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Bangun ulang kerangka model sebelum diisi bobot
model = models.mobilenet_v2(weights=None)
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model = model.to(DEVICE)
# ====================================================

# 1. Siapkan Dataloader TEST
test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'TEST'), data_transforms['val'])
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Muat bobot model terbaik (Pakai yang skor 68% tadi)
model.load_state_dict(torch.load('best_RespiScan_MobileNetV2_Scalogram.pth'))
model.eval()

# --- 3. TRIK CERDAS: LOAD METADATA DULUAN ---
print("📂 Memuat Metadata.csv untuk referensi ID pasien...")
path_metadata = r"C:\TBC SCREENING PENELITIAN\metadata_databaru\TBscreen_Dataset\Metadata.csv"
df_meta = pd.read_csv(path_metadata) 
df_meta.columns = df_meta.columns.str.strip()
df_meta['subject'] = df_meta['subject'].astype(str).str.strip()

# Ambil semua subject valid dari CSV
valid_subjects = sorted(df_meta['subject'].unique(), key=len, reverse=True)

# Fungsi untuk mendeteksi ID pasien di dalam nama file yang panjang
def ekstrak_subject_dari_nama(filename):
    for subj in valid_subjects:
        if subj in filename: 
            return subj
    return "TIDAK_KETEMU"

# 4. Prediksi Data TEST
results = []
file_paths = [path for path, label in test_dataset.samples]

print("🔍 Memulai prediksi pada data TEST dan mencocokkan ID...")
with torch.no_grad():
    start_idx = 0
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        for j in range(len(preds)):
            current_idx = start_idx + j
            nama_file_asli = os.path.basename(file_paths[current_idx])
            
            # --- GUNAKAN FUNGSI CERDAS KITA ---
            id_subject = ekstrak_subject_dari_nama(nama_file_asli)
            
            results.append({
                'subject': id_subject, 
                'Label_Asli': 'TB' if labels[j].item() == 1 else 'Healthy',
                'Tebakan_AI': 'TB' if preds[j].item() == 1 else 'Healthy',
                'Status_Benar': labels[j].item() == preds[j].item()
            })
        start_idx += len(preds)

df_prediksi = pd.DataFrame(results)

if (df_prediksi['subject'] == "TIDAK_KETEMU").any():
    print("⚠️ Ada beberapa file gambar yang ID-nya tidak terdaftar di Metadata.csv!")

# --- CEK JUMLAH PASIEN ASLI ---
print("\n" + "="*50)
print(f"🖼️ Total gambar (Scalogram) di data TEST : {len(df_prediksi)}")
pasien_unik = df_prediksi[df_prediksi['subject'] != 'TIDAK_KETEMU']['subject'].nunique()
print(f"🧍 Total ORANG (Pasien Unik) di data TEST: {pasien_unik}")
print("="*50 + "\n")

# 5. Gabungkan dengan Metadata menggunakan 'subject'
print("🔗 Menggabungkan data hasil prediksi dengan fitur klinis...")
df_analisis = pd.merge(df_prediksi, df_meta, on='subject', how='inner')

if len(df_analisis) == 0:
    print("⚠️ WARNING: Data gagal digabungkan (Merge kosong)!")
else:
    print(f"✅ Berhasil memetakan {len(df_analisis)} dari {len(df_prediksi)} tebakan AI ke profil pasien!\n")

    # 6. --- CETAK HASIL ANALISIS BIAS ---
    print("📊 HASIL ANALISIS BIAS (Akurasi AI per Kelompok Pasien):")
    print("-" * 50)

    def print_bias_stat(df, column_name, title):
        if column_name in df.columns:
            print(f"\n>> Akurasi berdasarkan {title}:")
            akurasi = df.groupby(column_name)['Status_Benar'].mean() * 100
            jumlah = df.groupby(column_name)['Status_Benar'].count()
            
            for index, value in akurasi.items():
                print(f"   - {index}: {value:.2f}% (dari {jumlah[index]} data)")

    print_bias_stat(df_analisis, 'gndr_a', 'Jenis Kelamin (Female/Male)')
    print_bias_stat(df_analisis, 'smkr_a', 'Status Perokok')
    print_bias_stat(df_analisis, 'Cavities - Y/N; if Y number of lobes', 'Kavitas Paru (Lubang)')
    print_bias_stat(df_analisis, 'lab_sptm_xprt_rslt_a', 'Hasil Lab GeneXpert (Beban Bakteri)')
    print("-" * 50)

C:\Users\Gibran\AppData\Local\Temp\ipykernel_27848\2227186374.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_RespiScan_MobileNet

📂 Memuat Metadata.csv untuk referensi ID pasien...
🔍 Memulai prediksi pada data TEST dan mencocokkan ID...

🖼️ Total gambar (Scalogram) di data TEST : 442
🧍 Total ORANG (Pasien Unik) di data TEST: 31

🔗 Menggabungkan data hasil prediksi dengan fitur klinis...
✅ Berhasil memetakan 442 dari 442 tebakan AI ke profil pasien!

📊 HASIL ANALISIS BIAS (Akurasi AI per Kelompok Pasien):
--------------------------------------------------

>> Akurasi berdasarkan Jenis Kelamin (Female/Male):
   - 0.0: 76.52% (dari 230 data)
   - 1.0: 76.79% (dari 112 data)

>> Akurasi berdasarkan Status Perokok:
   - 0.0: 78.60% (dari 257 data)
   - 1.0: 100.00% (dari 39 data)
   - 2.0: 16.67% (dari 30 data)

>> Akurasi berdasarkan Kavitas Paru (Lubang):
   - NO: 83.78% (dari 37 data)
   - YES - 1: 71.96% (dari 107 data)
   - YES - 2: 57.41% (dari 54 data)

>> Akurasi berdasarkan Hasil Lab GeneXpert (Beban Bakteri):
   - 2.0: 81.82% (dari 11 data)
   - 3.0: 48.28% (dari 29 data)
   - 4.0: 84.68% (dari 124 data)
   

In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

print("🏥 MENGHITUNG PATIENT-LEVEL ACCURACY (MAJORITY VOTE)")
print("=" * 50)

patient_results = []

# 1. Kelompokkan data berdasarkan ID Pasien (subject)
# Ingat: Kita abaikan dulu yang 'TIDAK_KETEMU' agar data valid
df_valid = df_prediksi[df_prediksi['subject'] != 'TIDAK_KETEMU']
grouped = df_valid.groupby('subject')

# 2. Lakukan Majority Vote untuk setiap pasien
for subject, group in grouped:
    # Ambil label asli pasien ini (semua gambar dari 1 pasien pasti labelnya sama)
    true_label = group['Label_Asli'].iloc[0]
    
    # Hitung suara terbanyak dari tebakan AI
    # .value_counts() otomatis mengurutkan dari yang paling sering muncul
    majority_vote = group['Tebakan_AI'].value_counts().index[0]
    
    # Simpan hasil rekapitulasi per pasien
    patient_results.append({
        'ID_Pasien': subject,
        'Total_Gambar': len(group),
        'Tebak_TB': (group['Tebakan_AI'] == 'TB').sum(),
        'Tebak_Healthy': (group['Tebakan_AI'] == 'Healthy').sum(),
        'Label_Asli': true_label,
        'Keputusan_AI': majority_vote,
        'Benar': true_label == majority_vote
    })

df_patient = pd.DataFrame(patient_results)

# 3. Tampilkan contoh dapur votingnya
print("\n📝 Contoh Dapur Voting per Pasien:")
print(df_patient[['ID_Pasien', 'Total_Gambar', 'Tebak_TB', 'Tebak_Healthy', 'Label_Asli', 'Keputusan_AI']].head())

# 4. Hitung Metrik Evaluasi Standar Medis
y_true = df_patient['Label_Asli']
y_pred = df_patient['Keputusan_AI']

accuracy = accuracy_score(y_true, y_pred)

# Ekstrak nilai True Negative, False Positive, False Negative, True Positive
cm = confusion_matrix(y_true, y_pred, labels=['Healthy', 'TB'])
tn, fp, fn, tp = cm.ravel()

# Hitung Sensitivity dan Specificity (Hindari pembagian dengan nol)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0 
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0 

print("\n" + "=" * 50)
print(" HASIL EVALUASI TINGKAT PASIEN ")
print("=" * 50)
print(f"Total Pasien Diuji   : {len(df_patient)} Orang")
print(f" Accuracy Total    : {accuracy * 100:.2f}%")
print(f" Sensitivity (TB)  : {sensitivity * 100:.2f}%")
print(f" Specificity (Sehat): {specificity * 100:.2f}%")
print("=" * 50)
print(" Keterangan:")
print("- Sensitivity: Dari semua pasien TB asli, berapa % yang sukses dideteksi AI?")
print("- Specificity: Dari semua pasien SEHAT asli, berapa % yang sukses dikenali AI?")

🏥 MENGHITUNG PATIENT-LEVEL ACCURACY (MAJORITY VOTE)

📝 Contoh Dapur Voting per Pasien:
  ID_Pasien  Total_Gambar  Tebak_TB  Tebak_Healthy Label_Asli Keputusan_AI
0  PID_108A            21        10             11         TB      Healthy
1   PID_10A            11         9              2         TB           TB
2   PID_10C            24         9             15    Healthy      Healthy
3   PID_11C             2         2              0    Healthy           TB
4   PID_12A             2         2              0         TB           TB

 HASIL EVALUASI TINGKAT PASIEN 
Total Pasien Diuji   : 31 Orang
 Accuracy Total    : 83.87%
 Sensitivity (TB)  : 88.89%
 Specificity (Sehat): 76.92%
 Keterangan:
- Sensitivity: Dari semua pasien TB asli, berapa % yang sukses dideteksi AI?
- Specificity: Dari semua pasien SEHAT asli, berapa % yang sukses dikenali AI?


In [3]:
print(f"Total gambar (Scalogram) di data TEST: {len(df_analisis)}")
print(f"Total ORANG (Pasien Unik) di data TEST: {df_analisis['subject'].nunique()}")

Total gambar (Scalogram) di data TEST: 442
Total ORANG (Pasien Unik) di data TEST: 31


In [1]:
import os

DATA_DIR = 'RespiScan_Train_Ready' 
splits = ['TRAIN', 'VAL', 'TEST']
classes = ['Healthy', 'TB']

print("📊 AUDIT DISTRIBUSI DATASET RESPISCAN")
print("=" * 40)

total_all = 0
distribusi = {}

for split in splits:
    total_split = 0
    print(f"📁 Folder: {split}")
    for cls in classes:
        # Pengecekan huruf besar/kecil otomatis
        path = os.path.join(DATA_DIR, split, cls)
        if not os.path.exists(path):
            path = os.path.join(DATA_DIR, split.lower(), cls)
            
        if os.path.exists(path):
            jumlah_file = len([name for name in os.listdir(path) if os.path.isfile(os.path.join(path, name))])
            print(f"   - {cls}: {jumlah_file} gambar")
            total_split += jumlah_file
        else:
            print(f"   - {cls}: 0 gambar (Folder tidak ditemukan)")
            
    print(f"   >> TOTAL {split}: {total_split} gambar\n")
    distribusi[split] = total_split
    total_all += total_split

print("=" * 40)
print(f"📈 TOTAL KESELURUHAN DATA: {total_all} gambar")

if total_all > 0:
    print("\n⚖️ PROPORSI PEMBAGIAN (SPLIT RATIO):")
    for split in splits:
        persentase = (distribusi[split] / total_all) * 100
        print(f"- {split}: {persentase:.1f}%")

📊 AUDIT DISTRIBUSI DATASET RESPISCAN
📁 Folder: TRAIN
   - Healthy: 1338 gambar
   - TB: 975 gambar
   >> TOTAL TRAIN: 2313 gambar

📁 Folder: VAL
   - Healthy: 298 gambar
   - TB: 269 gambar
   >> TOTAL VAL: 567 gambar

📁 Folder: TEST
   - Healthy: 244 gambar
   - TB: 198 gambar
   >> TOTAL TEST: 442 gambar

📈 TOTAL KESELURUHAN DATA: 3322 gambar

⚖️ PROPORSI PEMBAGIAN (SPLIT RATIO):
- TRAIN: 69.6%
- VAL: 17.1%
- TEST: 13.3%
